# Bachelor Thesis

© 2026 Yvan Richard   
University of St. Gallen, Spring Term 2026

## Reproduction of Barber et al. (2022)

---

## Foreword

In this notebook, the purpose is to reproduce Table I in Barber et al. (2022). The table depicts the summary statistics of the main data set.

## 1. Libraries & Data

I first load the relevant libraries and data.

In [6]:
# libs 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
import seaborn as sns
from datetime import datetime

In [7]:
# data
main = pd.read_csv("../../../data/processed/baseline_trading_sample.csv")

# parses dates
main["date"] = pd.to_datetime(main["date"], format="%Y-%m-%d")

/var/folders/7v/_v_y1jpx0rl056gg5rkjsw4r0000gn/T/ipykernel_76611/380983785.py:2: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  main = pd.read_csv("../../../data/processed/baseline_trading_sample.csv")


In [8]:
# unique tickers list
unique_tickers = set(main["ticker"].unique())
print(f"Number of unique tickers: {len(unique_tickers)}")

Number of unique tickers: 8272


Once the data are loaded, I build the relevant variables for computing the summary statistics.

In [9]:
# size
main['size'] = main['shrout'] * main['prc'].abs()

# build binary variable 'has_users' indicating whether there are any users for that ticker on that day
main["has_users"] = (main["users_close"] > 0).astype(int)

I now compute Panel A

In [10]:
# summary stats Panel A
summary_stats_panel_a = main[["users_close", "users_last", "userchg", "userratio", "prc", "size", "ret",
                                "daily_buys", "daily_sells", "net_buys", "taq_retimb"]].describe().T
print(summary_stats_panel_a)

                 count          mean           std            min  \
users_close  3833767.0  2.141724e+03  1.583332e+04       0.000000   
users_last   3841684.0  2.140932e+03  1.583070e+04       0.000000   
userchg      3803796.0  9.538341e+00  3.788631e+02 -548766.000000   
userratio    3763204.0  1.007797e+00  1.163591e+00       0.000000   
prc          3841684.0  4.015296e+01  9.199193e+01    -245.274990   
size         3766791.0  5.603107e+06  3.099671e+07      29.700000   
ret          3823495.0  4.484403e-04  4.959918e-02      -2.468715   
daily_buys   3659438.0  1.993166e+02  1.105930e+03       0.000000   
daily_sells  3659438.0  1.778382e+02  8.711857e+02       0.000000   
net_buys     3659438.0  2.147846e+01  3.653058e+02  -30215.000000   
taq_retimb   3659438.0  1.064730e-02  3.518729e-01      -1.000000   

                      25%            50%           75%           max  
users_close     40.000000     173.000000  7.130000e+02  9.904440e+05  
users_last      40.000000    

In [11]:
# Panel B: Daily Observations (Summed Variable Averaged across Days)
# keep only observations where has_users == 1
main = main[main["has_users"] == 1]

# group by date and sum the relevant variables: [users_close, users_last, userchg, daily_buys, daily_sells, net_buys]
daily_summary = main.groupby("date").agg({
    "users_close": "sum",
    "users_last": "sum",
    "userchg": "sum",
    "daily_buys": "sum",
    "daily_sells": "sum",
    "net_buys": "sum",
    "has_users": "sum"
}).reset_index()

# summary stats Panel B
summary_stats_panel_b = daily_summary[["users_close", "users_last", "userchg", "daily_buys", "daily_sells", "net_buys", "has_users"]].describe().T
print(summary_stats_panel_b)

             count          mean           std        min         25%  \
users_close  564.0  1.455828e+07  9.766320e+06  5348242.0  8105122.75   
users_last   564.0  1.456763e+07  9.776240e+06  5349883.0  8106857.00   
userchg      564.0  6.437185e+04  1.099128e+05  -185824.0    12952.00   
daily_buys   564.0  1.287187e+06  7.469847e+05   393553.0   829495.75   
daily_sells  564.0  1.148373e+06  5.947144e+05   365517.0   781605.00   
net_buys     564.0  1.388142e+05  1.711772e+05   -68884.0    38594.50   
has_users    564.0  6.724975e+03  6.201867e+02     4491.0     6195.75   

                    50%          75%         max  
users_close  11577055.5  14880844.75  41315823.0  
users_last   11579361.0  14886598.75  41328159.0  
userchg         22160.5     51820.25    817189.0  
daily_buys     932225.0   1310552.00   3972023.0  
daily_sells    880180.0   1194709.25   3200011.0  
net_buys        63553.5    135918.50   1007408.0  
has_users        6642.0      7461.00      7614.0  


In [22]:
# check most popular tickers
top_tickers = main.groupby("ticker")["users_close"].sum().sort_values(ascending=False).head(10)
print(top_tickers)

ticker
ACB     211168422.0
F       192038116.0
GE      182523030.0
AAPL    136557927.0
MSFT    134402176.0
GPRO    125694624.0
FIT     113681676.0
DIS     110533627.0
AMD      93902001.0
CRON     93409958.0
Name: users_close, dtype: float64
